In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
df =pd.read_csv("/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_metadata")

In [11]:
import os
part1_path = "/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_images_part_1"
part2_path = "/content/drive/MyDrive/skin-disease-detection-cnn/data/HAM10000_images_part_2"

In [12]:
def get_image_path(image_id):
    filename = image_id + ".jpg"
    path1 = os.path.join(part1_path, filename)
    if os.path.exists(path1):
        return path1
    else:
        return os.path.join(part2_path, filename)

In [13]:
unique_lesions = df['lesion_id'].unique()
unique_lesions
print(len(unique_lesions))

7470


In [14]:
lesion_labels = df.groupby('lesion_id')['dx'].first()
print(lesion_labels.head())
from sklearn.model_selection import train_test_split

train_lesions, temp_lesions = train_test_split(
    lesion_labels.index,
    test_size=0.30,
    stratify=lesion_labels.values,
    random_state=42
)

val_lesions, test_lesions = train_test_split(
    temp_lesions,
    test_size=0.50,
    random_state=42
)

print(len(train_lesions))
print(len(val_lesions))
print(len(test_lesions))

lesion_id
HAM_0000000     nv
HAM_0000001    bkl
HAM_0000002    mel
HAM_0000003     nv
HAM_0000004     nv
Name: dx, dtype: object
5229
1120
1121


In [15]:
# Train dataframe
train_df = df[df['lesion_id'].isin(train_lesions)].copy()

# Validation dataframe
val_df = df[df['lesion_id'].isin(val_lesions)].copy()

# Test dataframe
test_df = df[df['lesion_id'].isin(test_lesions)].copy()

# Shapes print
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# Train class distribution
print(train_df['dx'].value_counts(normalize=True))

# Validation class distribution
print(val_df['dx'].value_counts(normalize=True))

# Test class distribution
print(test_df['dx'].value_counts(normalize=True))

Train shape: (6981, 8)
Validation shape: (1518, 8)
Test shape: (1516, 8)
dx
nv       0.670821
mel      0.110729
bkl      0.110586
bcc      0.051712
akiec    0.031801
vasc     0.014181
df       0.010170
Name: proportion, dtype: float64
dx
nv       0.681159
bkl      0.102767
mel      0.099473
bcc      0.056653
akiec    0.031621
df       0.016469
vasc     0.011858
Name: proportion, dtype: float64
dx
nv       0.651715
mel      0.124670
bkl      0.112797
bcc      0.044195
akiec    0.037599
vasc     0.016491
df       0.012533
Name: proportion, dtype: float64


In [16]:
# Import LabelEncoder for converting text labels into numeric labels
from sklearn.preprocessing import LabelEncoder

# Create the encoder object
encoder = LabelEncoder()

# Fit the encoder only on training labels
# This allows the model to learn all unique classes from training data
encoder.fit(train_df['dx'])

# Convert text labels into numeric labels for training data
train_df['label'] = encoder.transform(train_df['dx'])

# Apply the same learned mapping on validation data
val_df['label'] = encoder.transform(val_df['dx'])

# Apply the same learned mapping on test data
test_df['label'] = encoder.transform(test_df['dx'])

# Print all classes learned by the encoder
print(encoder.classes_)

['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']
